# Floci Lakehouse Demo with Raspberry Pi Sense HAT, Java 25, Pi4J, Spark, Hudi, Delta Lake, and Iceberg

This notebook reads live temperature values from a Raspberry Pi **Sense HAT** using **Java 25** and **Pi4J Drivers**, then writes the readings to a local S3-compatible Floci endpoint using Apache Spark.

The demo covers:

- Reading temperature from Sense HAT
- Creating a Spark DataFrame
- Writing and reading Apache Parquet
- Writing and reading Apache Hudi
- Writing and reading Delta Lake
- Writing and reading Apache Iceberg

Expected runtime environment:

- Raspberry Pi 4, 64-bit OS
- Sense HAT attached
- I2C enabled
- Docker container running with access to `/dev/i2c-1`
- Java 25
- Jupyter Java kernel
- Floci running as `http://floci:4566`


## 1. Maven Dependencies

The Pi4J Sense HAT driver is provided by `com.pi4j:pi4j-drivers`.

This notebook also loads the lakehouse dependencies used by Spark:

- `hadoop-aws`
- `aws-java-sdk-bundle`
- `delta-spark`
- `hudi-spark3.5-bundle`
- `iceberg-spark-runtime-3.5`


In [ ]:
%maven com.pi4j:pi4j-core:5.0.0-SNAPSHOT
%maven com.pi4j:pi4j-plugin-ffm:5.0.0-SNAPSHOT
%maven com.pi4j:pi4j-drivers:1.1.0

%maven org.apache.hadoop:hadoop-aws:3.3.4
%maven com.amazonaws:aws-java-sdk-bundle:1.12.262
%maven software.amazon.awssdk:s3:2.25.60

%maven io.delta:delta-spark_2.12:3.2.0
%maven org.apache.hudi:hudi-spark3.5-bundle_2.12:1.0.2
%maven org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1


## 1b. Bootstrap S3 Buckets on Floci

Create the buckets used by the lakehouse writes before Spark runs. Spark's S3A client does not create buckets on demand.


In [ ]:
import java.net.URI;

import software.amazon.awssdk.auth.credentials.AwsBasicCredentials;
import software.amazon.awssdk.auth.credentials.StaticCredentialsProvider;
import software.amazon.awssdk.regions.Region;
import software.amazon.awssdk.services.s3.S3Client;
import software.amazon.awssdk.services.s3.model.BucketAlreadyExistsException;
import software.amazon.awssdk.services.s3.model.BucketAlreadyOwnedByYouException;
import software.amazon.awssdk.services.s3.model.CreateBucketRequest;

URI flociEndpoint = URI.create("http://floci:4566");

S3Client bootstrapS3 = S3Client.builder()
    .endpointOverride(flociEndpoint)
    .region(Region.US_EAST_1)
    .credentialsProvider(StaticCredentialsProvider.create(
        AwsBasicCredentials.create("test", "test")))
    .forcePathStyle(true)
    .build();

for (String bucket : java.util.List.of(
        "iot-raw", "iot-hudi", "iot-delta", "iot-iceberg")) {
    try {
        bootstrapS3.createBucket(CreateBucketRequest.builder().bucket(bucket).build());
        System.out.println("Created bucket: " + bucket);
    } catch (BucketAlreadyOwnedByYouException | BucketAlreadyExistsException e) {
        System.out.println("Bucket already exists: " + bucket);
    }
}

bootstrapS3.close();


## 2. Check Java Version

This notebook is intended to run with Java 25 because `pi4j-drivers` is built for Java 25.


In [ ]:
System.out.println("Java version: " + System.getProperty("java.version"));
System.out.println("Java vendor: " + System.getProperty("java.vendor"));


## 3. Read Temperature from Sense HAT

The Sense HAT exposes temperature via two chips (the humidity sensor and the pressure sensor). Neither reading is authoritative on its own — averaging them is a common convenience.


In [ ]:
import com.pi4j.Pi4J;
import com.pi4j.context.Context;
import com.pi4j.drivers.hat.raspberry.SenseHat;

Context pi4j = Pi4J.newAutoContext();

SenseHat senseHat = new SenseHat(pi4j);

double tempFromHumidity = senseHat.getTemperatureFromHumidity();
double tempFromPressure = senseHat.getTemperatureFromPressure();
double currentTemperature = (tempFromHumidity + tempFromPressure) / 2.0;

double currentHumidity = senseHat.getHumidity();
double currentPressure = senseHat.getPressure();

System.out.printf("Temperature: %.2f C%n", currentTemperature);
System.out.printf("Humidity: %.2f %%RH%n", currentHumidity);
System.out.printf("Pressure: %.2f hPa%n", currentPressure);


## 4. Collect Multiple Sensor Readings

For the lakehouse demo, collect a small batch of readings from the Sense HAT.


In [ ]:
import java.sql.Timestamp;
import java.time.Instant;
import java.util.ArrayList;
import java.util.List;

record TemperatureReading(
    String deviceId,
    Timestamp eventTime,
    double temperatureCelsius,
    double humidityPercent,
    double pressureHpa
) {}

String deviceId = "raspberry-pi-4-sensehat-01";

List<TemperatureReading> readings = new ArrayList<>();

for (int i = 0; i < 10; i++) {
    double t = (senseHat.getTemperatureFromHumidity()
              + senseHat.getTemperatureFromPressure()) / 2.0;

    readings.add(new TemperatureReading(
        deviceId,
        Timestamp.from(Instant.now()),
        t,
        senseHat.getHumidity(),
        senseHat.getPressure()
    ));

    Thread.sleep(1000);
}

readings.forEach(System.out::println);


## 5. Create Spark Session

The Spark session uses:

- Spark local mode
- Floci S3 endpoint: `http://floci:4566`
- S3 path-style access
- Delta Lake extensions
- Iceberg Hadoop catalog


In [ ]:
import org.apache.spark.sql.SparkSession;

SparkSession spark = SparkSession.builder()
    .appName("floci-sensehat-java-lakehouse")
    .master("local[*]")

    // Hudi requires Kryo
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")

    // S3A / Floci
    .config("spark.hadoop.fs.s3a.endpoint", "http://floci:4566")
    .config("spark.hadoop.fs.s3a.access.key", "test")
    .config("spark.hadoop.fs.s3a.secret.key", "test")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

    // Delta Lake + Iceberg + Hudi extensions
    .config("spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension,"
        + "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
        + "org.apache.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    // Iceberg
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://iot-iceberg/warehouse")

    .getOrCreate();

spark.sparkContext().setLogLevel("WARN");

System.out.println("Spark version: " + spark.version());


## 6. Create a DataFrame from Sense HAT Readings

Java `record` accessors (`deviceId()`) do not match the JavaBean convention (`getDeviceId()`) that Spark's bean encoder expects. Build the DataFrame from `Row` + an explicit `StructType` instead.


In [ ]:
import java.util.stream.Collectors;

import org.apache.spark.sql.Row;
import org.apache.spark.sql.RowFactory;
import org.apache.spark.sql.types.DataTypes;
import org.apache.spark.sql.types.StructType;
import org.apache.spark.sql.functions;

StructType schema = new StructType()
    .add("deviceId", DataTypes.StringType, false)
    .add("eventTime", DataTypes.TimestampType, false)
    .add("temperatureCelsius", DataTypes.DoubleType, false)
    .add("humidityPercent", DataTypes.DoubleType, false)
    .add("pressureHpa", DataTypes.DoubleType, false);

List<Row> rows = readings.stream()
    .map(r -> RowFactory.create(
        r.deviceId(),
        r.eventTime(),
        r.temperatureCelsius(),
        r.humidityPercent(),
        r.pressureHpa()))
    .collect(Collectors.toList());

var df = spark.createDataFrame(rows, schema)
    .withColumn("eventDate", functions.to_date(functions.col("eventTime")));

df.printSchema();
df.show(false);


## 7. Write and Read Parquet

Parquet is the physical columnar file format. Hudi, Delta Lake, and Iceberg all use Parquet as a common storage format underneath.


In [ ]:
String parquetPath = "s3a://iot-raw/sensehat/temperature_parquet";

df.write()
  .mode("overwrite")
  .parquet(parquetPath);

var parquetDf = spark.read()
  .parquet(parquetPath);

parquetDf.show(false);


## 8. Write and Read Apache Hudi

Hudi adds table-level capabilities such as commits, upserts, and incremental processing on top of Parquet files.


In [ ]:
String hudiPath = "s3a://iot-hudi/sensehat/temperature_hudi";

df.write()
  .format("hudi")
  .option("hoodie.table.name", "temperature_hudi")
  .option("hoodie.datasource.write.recordkey.field", "deviceId,eventTime")
  .option("hoodie.datasource.write.precombine.field", "eventTime")
  .option("hoodie.datasource.write.partitionpath.field", "eventDate")
  .option("hoodie.datasource.write.keygenerator.class",
          "org.apache.hudi.keygen.ComplexKeyGenerator")
  .option("hoodie.datasource.write.operation", "upsert")
  .mode("overwrite")
  .save(hudiPath);

var hudiDf = spark.read()
  .format("hudi")
  .load(hudiPath);

hudiDf.show(false);


## 9. Write and Read Delta Lake

Delta Lake adds a transaction log to Parquet data and enables table operations such as updates, deletes, merges, and time travel.


In [ ]:
String deltaPath = "s3a://iot-delta/sensehat/temperature_delta";

df.write()
  .format("delta")
  .mode("overwrite")
  .save(deltaPath);

var deltaDf = spark.read()
  .format("delta")
  .load(deltaPath);

deltaDf.show(false);

// Generate a symlink-format manifest so Athena/Presto (via SymlinkTextInputFormat)
// can read the same Delta table without a native Delta reader.
io.delta.tables.DeltaTable
    .forPath(spark, deltaPath)
    .generate("symlink_format_manifest");


## 10. Write and Read Apache Iceberg

Iceberg provides a table format with metadata, snapshots, schema evolution, partition evolution, and engine interoperability.


In [ ]:
df.createOrReplaceTempView("sensehat_readings");

spark.sql("DROP TABLE IF EXISTS local.default.temperature_iceberg");

spark.sql("""
CREATE TABLE local.default.temperature_iceberg (
  deviceId STRING,
  eventTime TIMESTAMP,
  temperatureCelsius DOUBLE,
  humidityPercent DOUBLE,
  pressureHpa DOUBLE
)
USING iceberg
""");

spark.sql("""
INSERT INTO local.default.temperature_iceberg
SELECT deviceId, eventTime, temperatureCelsius, humidityPercent, pressureHpa
FROM sensehat_readings
""");

spark.sql("SELECT * FROM local.default.temperature_iceberg").show(false);


## 11. Cleanup

Close Pi4J resources when finished.


In [ ]:
try {
    senseHat.close();
} catch (Exception e) {
    System.out.println("Sense HAT close warning: " + e.getMessage());
}

try {
    pi4j.shutdown();
} catch (Exception e) {
    System.out.println("Pi4J shutdown warning: " + e.getMessage());
}

try {
    spark.stop();
} catch (Exception e) {
    System.out.println("Spark stop warning: " + e.getMessage());
}

System.out.println("Done.");


## Troubleshooting

### Sense HAT cannot be detected

Make sure I2C is enabled on the Raspberry Pi:

```bash
sudo raspi-config
```

Then enable:

```text
Interface Options -> I2C
```

Reboot after enabling I2C.

### Docker container cannot access Sense HAT

The Jupyter container needs access to the I2C device. Add this to the `jupyter` service in `docker-compose.yml`:

```yaml
devices:
  - "/dev/i2c-1:/dev/i2c-1"
privileged: true
```

For LED matrix access you may also need framebuffer access:

```yaml
devices:
  - "/dev/i2c-1:/dev/i2c-1"
  - "/dev/fb0:/dev/fb0"
```

### Java version is not 25

Use a custom Dockerfile based on `jupyter/all-spark-notebook` and install a Java 25 JDK, then configure `JAVA_HOME`.
